In [0]:
%run ../init_framework

In [0]:
# 1. Initialize the CDF Read Stream  
df_bronze_loans = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1) # Required for the very first execution
    .table(LOANS_BRONZE))

In [0]:
LOAN_RENAME_MAP = {
    "loan_amnt": "loan_amount",
    "funded_amnt": "funded_amount",
    "term": "loan_term_months",
    "int_rate": "interest_rate",
    "installment": "monthly_installment",
    "issue_d": "issue_date",
    "loan_status": "loan_status",
    "purpose": "loan_purpose"
}

# --- Execution in Notebook ---
df_renamed_loans = standardize_column_names(df_bronze_loans, LOAN_RENAME_MAP, strict=True)

In [0]:
df_with_metadata = apply_silver_metadata(df_renamed_loans)

In [0]:
df_distinct = df_with_metadata.distinct()

In [0]:
# Keep Valid Loans: Drop NULL or UNKNOWN Loans
col_ls = ["loan_amount", "funded_amount", "loan_term_months", "interest_rate", "monthly_installment", "issue_date", "loan_status", "loan_purpose"]
df_valid_loans = drop_critical_nulls(df_distinct, col_ls)

In [0]:
# Convert loan_term_months to loan_term_years and cast as INT

from pyspark.sql.functions import col, regexp_replace

df_loan_years = df_valid_loans.withColumn(
    "loan_term_years", 
    (regexp_replace(col("loan_term_months"), r"\D+", "").cast("int") / 12).cast("int")
).drop("loan_term_months")

In [0]:
from pyspark.sql.functions import col, coalesce, lit, broadcast

# Read the lookup table for valid codes
df_lkp = spark.read.table(REF_LOAN_PURPOSES).filter("is_active = true")

# Join with aliases to manage the "loan_purpose" column from both sides
df_joined = df_loan_years.alias("base").join(
    broadcast(df_lkp.alias("lkp")), 
    col("base.loan_purpose") == col("lkp.loan_purpose"), 
    "left"
).drop(col("base.loan_purpose"))

# 3. Simple Logic: If is_active is NULL, the purpose wasn't in our active list
df_loanspurpose_cleaned = (df_joined
    .withColumn("loan_purpose", coalesce(col("lkp.loan_purpose"), lit("other")))
    .drop("is_active")
)

In [0]:
# Final deduplication on the primary key to prevent MERGE conflicts
df_loans_final = df_loanspurpose_cleaned.dropDuplicates(["loan_id"])

In [0]:
def upsert_loans_to_silver(microBatchDF, batchId):
    """
    Stateless MERGE from Bronze CDF into Silver Loans using explicit SQL mapping.
    """
    spark_session = microBatchDF.sparkSession

    # Register the incoming micro-batch as a Temp View
    microBatchDF.createOrReplaceTempView("loans_batch_updates")

    # Execute explicit SQL MERGE. 
    # NOTE: Joined strictly on loan_id. 
    merge_query = f"""
    MERGE INTO {LOANS_SILVER} AS target
        USING loans_batch_updates AS source
        ON target.loan_id = source.loan_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.member_id = source.member_id,
            target.loan_amount = source.loan_amount,
            target.funded_amount = source.funded_amount,
            target.interest_rate = source.interest_rate,
            target.monthly_installment = source.monthly_installment,
            target.issue_date = source.issue_date,
            target.loan_status = source.loan_status,
            target.title = source.title,
            target.loan_term_years = source.loan_term_years,
            target.loan_purpose = source.loan_purpose,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at
        WHEN NOT MATCHED THEN
          INSERT (
            loan_id, member_id, loan_amount, funded_amount, interest_rate, 
            monthly_installment, issue_date, loan_status, title, 
            loan_term_years, loan_purpose, 
            _ingested_at, _source_file, _bronze_version, _updated_at
          )
          VALUES (
            source.loan_id, source.member_id, source.loan_amount, source.funded_amount, source.interest_rate, 
            source.monthly_installment, source.issue_date, source.loan_status, source.title, 
            source.loan_term_years, source.loan_purpose, 
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at
          )
    """
    spark_session.sql(merge_query)

# Configure shuffle partitions for local cluster core count
spark.conf.set("spark.sql.shuffle.partitions", "16")
 
# --- Start Streaming Pipeline ---
query_loans = (
    df_loans_final.writeStream
    .outputMode("append") # Required for stateless CDF reads
    .option("checkpointLocation", SILVER_CHECKPOINT_LOANS)
    .trigger(availableNow=True)
    .foreachBatch(upsert_loans_to_silver)
    .start()
)

# Block cluster exit until this stream finishes
query_loans.awaitTermination()

In [0]:
%sql
select count(1) from lending_risk.silver.loans;
select * from lending_risk.silver.loans limit 3;
-- 1507112